# TraceIQ Advanced — Multi-Source RAG with Knowledge Routing

This notebook extends Notebook 1 by searching across multiple knowledge sources at the same time.

You can give it:
- **GitHub repository URLs** — it clones the repo and parses code using Tree-sitter (AST-aware)
- **PDF documents** — same semantic chunking as Notebook 1
- **Plain text / notes** — paste anything and it gets indexed too

Then you ask one question and it searches all sources together.

**What is new compared to Notebook 1:**

| Feature | Notebook 1 | This notebook |
|---------|-----------|---------------|
| Sources | Single PDF | GitHub repos + PDFs + text |
| Code chunking | Not supported | AST-based (Tree-sitter keeps functions intact) |
| Routing | None | Knowledge Router sends query to the right source |
| Caching | None | Semantic cache returns instant answers for repeated queries |

**How to run:** add sources using the sidebar inputs in the Gradio interface, click Build Index, then ask questions.

**API key needed:** `GROQ_API_KEY` in Colab Secrets.


## Step 1 — Install Dependencies

This installs all the libraries the notebook needs.

Run this cell once per Colab session. It only needs to run again if you restart the runtime.


In [ ]:
!pip install -q groq qdrant-client rank-bm25 pypdf gitpython gradio tree-sitter==0.20.4 tree-sitter-languages numpy scikit-learn sentence-transformers tiktoken semantic_text_splitter
print(" All packages installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 16.2 MB/s eta 0:00:00
 All packages installed.


## Step 2 — Imports and Configuration

This cell loads all the Python libraries and sets the constants used throughout the notebook.

Key settings defined here:
- `VECTOR_DIM` — size of the embedding vectors (384 for BGE-small)
- `TOP_K` — how many chunks to retrieve before reranking
- `CONTEXT_BUDGET` — maximum tokens sent to the LLM per query


In [ ]:
import os, re, uuid, shutil, tempfile, time
import numpy as np
from typing import List, Dict, Tuple, Optional, Any
from pathlib import Path
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer
from sentence_transformers import CrossEncoder
import tiktoken
from sklearn.preprocessing import normalize
import gradio as gr

CONFIG = {
    "vector_dim"        : 384,
    "qdrant_collection" : "traceiq_adv",
    "rrf_k"             : 60,     # RRF smoothing constant
    "top_k_retrieve"    : 12,     # candidates per retriever
    "top_k_final"       : 4,      # chunks sent to LLM (prevents context rot)
    "chunk_size_tokens" : 400,    # word-window size for prose
    "chunk_overlap"     : 50,
    "max_repo_files"    : 120,    # safety cap on repo files
    "groq_model"        : "llama-3.3-70b-versatile",
    "context_budget": 3000,
    "cross_encoder_model":
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "embedding_model":
    "BAAI/bge-small-en-v1.5",
}
print(" Config ready.")

 Config ready.


## Step 3 — LLM Setup

This cell connects to Groq and sets up the language model that generates the final answer.

The model only generates answers from the context we give it. It does not answer from memory.

If your `GROQ_API_KEY` is set in Colab Secrets, this runs automatically.


In [ ]:
def setup_groq_key() -> str:
    """Try Colab secrets  env var  interactive input."""
    try:
        from google.colab import userdata
        raw = userdata.get("GROQ_API_KEY")
        key = str(raw).strip() if not isinstance(raw, dict) else (
            raw.get("data", {}).get("payload") or raw.get("payload") or ""
        )
        if key:
            print(" Groq key loaded from Colab secrets.")
            return key
    except Exception:
        pass
    key = os.environ.get("GROQ_API_KEY", "").strip()
    if key:
        print(" Groq key loaded from environment.")
        return key
    key = input("Paste your Groq API key (get free at https://console.groq.com): ").strip()
    return key

_groq_client = None

def get_groq_client(api_key: str):
    from groq import Groq
    global _groq_client
    if _groq_client is None:
        _groq_client = Groq(api_key=api_key)
    return _groq_client

def llm_call(prompt: str, api_key: str) -> str:

    try:

        client = get_groq_client(api_key)

        resp = client.chat.completions.create(
            model=CONFIG["groq_model"],
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a retrieval-augmented AI assistant. "
                        "Answer using only the provided context."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.15,
            max_tokens=1024,
        )

        return resp.choices[0].message.content.strip()

    except Exception as e:

        print(f"LLM Error: {e}")

        return (
            "The language model request failed. "
            "Please try again."
        )

print(" LLM helpers ready.")

 LLM helpers ready.


## Step 4 — Embedding Model

This cell loads the embedding model that converts text into vectors.

I used `BAAI/bge-small-en-v1.5` because it gives strong retrieval quality while being small enough to run on CPU without slowing things down.

The same model is used for both indexing (when we store chunks) and querying (when we search for answers). This consistency is important — you cannot mix different embedding models.


In [ ]:
#Local Embedding Engine
class EmbedEngine:

    """
    BGE embedding engine.

    Uses:
    BAAI/bge-small-en-v1.5

    Output Dimension:
    384
    """

    def __init__(
        self,
        model_name="BAAI/bge-small-en-v1.5"
    ):

        self.model_name = model_name

        self.model = SentenceTransformer(
            model_name
        )

        self.actual_dim = 384

    def fit(self, texts):

        # kept for compatibility
        return self

    def embed(self, texts):

        return np.array(
            self.model.encode(
                texts,
                normalize_embeddings=True,
                show_progress_bar=False
            )
        )

    def embed_one(self, text):

        return self.embed([text])[0]

print("BGE EmbedEngine ready.")

BGE EmbedEngine ready.


## Step 5 — Code Parsing with Tree-sitter (AST Chunking)

This is the most important difference between this notebook and Notebook 1.

For code files, splitting at a fixed number of lines often cuts a function in half:

```
# Bad — function split across two chunks
Chunk 1: "def authenticate_user(username,"
Chunk 2: "password): return generate_token()"
```

Tree-sitter reads the actual structure of the code and keeps complete units together:

```
# Good — whole function in one chunk
Chunk 1: "def authenticate_user(username, password):
              token = generate_token()
              return token"
```

This makes search results far more useful. A complete function is a meaningful unit. Half a function is not.

Supported: Python, JavaScript, TypeScript.


In [ ]:
from tree_sitter_languages import get_parser

# Define AST node types to extract full functions and classes instead of partial code blocks.
_TOP_LEVEL_TYPES = {
    "python"     : {"function_definition", "class_definition", "decorated_definition",
                    "async_function_definition"},
    "javascript" : {"function_declaration", "class_declaration", "method_definition",
                    "arrow_function", "export_statement"},
    "typescript" : {"function_declaration", "class_declaration", "method_definition",
                    "interface_declaration", "type_alias_declaration",
                    "enum_declaration", "export_statement",
                    "abstract_class_declaration"},
}

LANG_MAP = {
    ".py"  : "python",
    ".js"  : "javascript",
    ".jsx" : "javascript",
    ".mjs" : "javascript",
    ".cjs" : "javascript",
    ".ts"  : "typescript",
    ".tsx" : "typescript",
    ".d.ts": "typescript",
}

def _node_text(node, src: bytes) -> str:
    return src[node.start_byte:node.end_byte].decode("utf-8", errors="replace")

def _extract_name(node, src: bytes) -> str:
    for child in node.children:
        if child.type in ("identifier", "property_identifier", "type_identifier"):
            return src[child.start_byte:child.end_byte].decode("utf-8", errors="replace")
    return "anonymous"

def _fallback_line_chunks(text: str, window: int = 60, stride: int = 40) -> List[Dict]:
    lines = text.splitlines()
    chunks = []
    for start in range(0, len(lines), stride):
        end = min(start + window, len(lines))
        chunk_text = "\n".join(lines[start:end]).strip()
        if len(chunk_text.split()) < 5:
            continue
        chunks.append({
            "content"    : chunk_text,
            "start_line" : start + 1,
            "end_line"   : end,
            "node_type"  : "text_window",
            "name"       : f"lines_{start+1}_{end}",
        })
        if end == len(lines):
            break
    return chunks

def ast_chunk_code(source_code: str, language: str) -> List[Dict[str, Any]]:
    """
    Split source into semantic chunks using Tree-sitter AST.
    Returns list of dicts: {content, start_line, end_line, node_type, name}
    """
    lang_key = language.lower()
    if lang_key not in _TOP_LEVEL_TYPES:
        return _fallback_line_chunks(source_code)
    try:
        parser = get_parser(lang_key)
        src_bytes = source_code.encode("utf-8", errors="replace")
        tree = parser.parse(src_bytes)
        chunks = []
        top_types = _TOP_LEVEL_TYPES[lang_key]

        def walk(node):
            if node.type in top_types:
                text = _node_text(node, src_bytes).strip()
                if len(text.split()) < 5:
                    return
                chunks.append({
                  "content": text,
                  "name": _extract_name(node, src_bytes),
                  "node_type": node.type,
                  "language": language,
                  "start_line": node.start_point[0] + 1,
                  "end_line": node.end_point[0] + 1,
              })
                return  # don't recurse into captured node
            for child in node.children:
                walk(child)

        walk(tree.root_node)
        return chunks if chunks else _fallback_line_chunks(source_code)
    except Exception:
        return _fallback_line_chunks(source_code)

print(" Tree-sitter AST chunker ready. Languages: Python, JavaScript, TypeScript")

 Tree-sitter AST chunker ready. Languages: Python, JavaScript, TypeScript


## Step 6 — PDF and Text Chunking

For non-code sources (PDFs and plain text), semantic chunking is used.

Text is split where the meaning changes rather than at a fixed word count. This keeps related sentences in the same chunk, which improves retrieval quality.

Each chunk keeps metadata — document name, page number, source type — so the answer can cite exactly where the information came from.


In [ ]:
import tiktoken
from semantic_text_splitter import TextSplitter

enc = tiktoken.get_encoding("cl100k_base")

def estimate_tokens(text: str) -> int:
    return len(enc.encode(text))

def chunk_text(
    text: str,
    max_tok: int = None
) -> List[str]:

    max_tok = max_tok or CONFIG["chunk_size_tokens"]

    splitter = TextSplitter(
        capacity=max_tok
    )

    chunks = splitter.chunks(text)

    return [
        c.strip()
        for c in chunks
        if len(c.split()) > 5
    ]

def parse_pdf(path: str) -> List[Dict]:

    pages = []

    reader = PdfReader(path)

    for i, page in enumerate(
        reader.pages,
        start=1
    ):

        text = (
            page.extract_text() or ""
        ).strip()

        if text:

            pages.append({
                "page": i,
                "text": text
            })

    return pages

print("Semantic PDF/Text chunker ready.")

Semantic PDF/Text chunker ready.


## Step 7 — Ingestion Pipeline

This is where all three source types come together.

For a GitHub repo:
1. Clone the repository
2. Walk every file
3. Skip noise folders (`node_modules`, `.git`, `build`, etc.)
4. Apply AST chunking to code files
5. Attach metadata to every chunk

For a PDF:
1. Extract text page by page
2. Apply semantic chunking
3. Attach page number and document name

For plain text:
1. Apply semantic chunking
2. Store as a separate knowledge object

All chunks go into the same list and get indexed together.


In [ ]:
SKIP_DIRS = {
    ".git", "__pycache__", ".venv", "venv",
    "node_modules", "dist", "build",
    ".next", "coverage", ".pytest_cache",
    "test", "tests", "spec", "__tests__",
}


def ingest_github_repo(repo_url: str) -> List[Dict]:

    import git

    chunks = []

    repo_name = (
        Path(repo_url)
        .stem
        .replace(".git", "")
    )

    tmp = tempfile.mkdtemp(
        prefix="traceiq_repo_"
    )

    try:

        print(f"Cloning {repo_url}")

        git.Repo.clone_from(
            repo_url,
            tmp,
            depth=1
        )

        repo_path = Path(tmp)

        file_count = 0

        for fpath in sorted(
            repo_path.rglob("*")
        ):

            if not fpath.is_file():
                continue

            if any(
                part in SKIP_DIRS
                for part in fpath.parts
            ):
                continue

            name = fpath.name.lower()

            if name.endswith(".d.ts"):
                ext = ".d.ts"
            else:
                ext = fpath.suffix.lower()

            if ext not in LANG_MAP:
                continue

            if file_count >= CONFIG["max_repo_files"]:
                break

            try:
                code = fpath.read_text(
                    encoding="utf-8",
                    errors="ignore"
                )
            except Exception:
                continue

            lang = LANG_MAP[ext]

            ast_chunks = ast_chunk_code(
                code,
                lang
            )

            rel_path = str(
                fpath.relative_to(repo_path)
            )

            for c in ast_chunks:

                chunks.append({
                    "id": str(uuid.uuid4()),

                    "source": f"repo:{rel_path}",

                    "repo": repo_name,

                    "file": rel_path,

                    "language": lang,

                    "type": "code",

                    "node_type": c["node_type"],

                    "name": c["name"],

                    "lines":
                        f"{c['start_line']}-{c['end_line']}",

                    "content": c["content"],

                    "tokens":
                        estimate_tokens(
                            c["content"]
                        ),
                })

            file_count += 1

        print(
            f"Parsed {file_count} files | "
            f"{len(chunks)} chunks"
        )

    finally:

        shutil.rmtree(
            tmp,
            ignore_errors=True
        )

    return chunks


def ingest_pdfs(
    pdf_paths: List[str]
) -> List[Dict]:

    chunks = []

    for pdf_path in pdf_paths:

        fname = Path(pdf_path).name

        document_id = str(uuid.uuid4())

        print(f"Parsing PDF: {fname}")

        pages = parse_pdf(pdf_path)

        for p in pages:

            page_chunks = chunk_text(
                p["text"]
            )

            for i, ct in enumerate(
                page_chunks,
                start=1
            ):

                chunks.append({

                    "id":
                        str(uuid.uuid4()),

                    "document_id":
                        document_id,

                    "document":
                        fname,

                    "page":
                        p["page"],

                    "source":
                        f"pdf:{fname}",

                    "file":
                        fname,

                    "language":
                        "text",

                    "type":
                        "pdf",

                    "node_type":
                        "paragraph",

                    "name":
                        f"p{p['page']}-s{i}",

                    "lines":
                        f"page {p['page']}",

                    "content":
                        ct,

                    "tokens":
                        estimate_tokens(ct),
                })

        print(
            f"{len(pages)} pages | "
            f"{len(chunks)} total chunks"
        )

    return chunks


def ingest_text(
    text: str,
    label: str = "text"
) -> List[Dict]:

    chunks = []

    for i, ct in enumerate(
        chunk_text(text),
        start=1
    ):

        chunks.append({

            "id":
                str(uuid.uuid4()),

            "source":
                f"{label}:pasted",

            "file":
                label,

            "language":
                "text",

            "type":
                "text",

            "node_type":
                "section",

            "name":
                f"s{i}",

            "lines":
                f"section {i}",

            "content":
                ct,

            "tokens":
                estimate_tokens(ct),
        })

    return chunks


print("Multi-source ingestor ready.")

Multi-source ingestor ready.


## Step 8 — Building the Search Index

After ingestion, two indexes are built:

**Vector index (Qdrant):**
Every chunk is embedded with BGE and stored in Qdrant. This enables semantic similarity search — finding chunks that mean the same thing as the query, even if different words are used.

**BM25 index:**
All chunks are also indexed with BM25 keyword search. This finds exact term matches — useful for API names, function names, and technical identifiers that semantic search can miss.

Having both indexes ready means the retrieval step can use both methods and combine their results.


In [ ]:
def build_index(all_chunks):

    print(
        f"\nBuilding index for "
        f"{len(all_chunks)} chunks ..."
    )

    texts = [
        c["content"]
        for c in all_chunks
    ]

    print("Loading BGE embedder...")

    emb = EmbedEngine()

    print("Embedding chunks...")

    vecs = emb.embed(texts)

    print("Building Qdrant...")

    qc = QdrantClient(
        location=":memory:"
    )

    qc.create_collection(
        collection_name=
            CONFIG["qdrant_collection"],

        vectors_config=
            VectorParams(
                size=CONFIG["vector_dim"],
                distance=Distance.COSINE
            ),
    )

    qc.upsert(
        collection_name=
            CONFIG["qdrant_collection"],

        points=[
            PointStruct(
                id=i,
                vector=vecs[i].tolist(),
                payload=c
            )
            for i, c in enumerate(all_chunks)
        ]
    )

    for i, c in enumerate(all_chunks):
        c["_idx"] = i

    print("Qdrant ready.")

    print("Building BM25...")

    bm25 = BM25Okapi([
        c["content"]
        .lower()
        .split()
        for c in all_chunks
    ])

    print("BM25 ready.")

    total_tok = sum(
        c["tokens"]
        for c in all_chunks
    )

    print(
        f"\nIndex ready:"
        f" {len(all_chunks)} chunks"
        f" | ~{total_tok:,} tokens\n"
    )

    return emb, qc, bm25

## Step 9 — Knowledge Routing, Hybrid Retrieval, and Reranking

**Step 9a — Knowledge Router**

Before searching, the router looks at the question and decides which source type is most relevant.

```
"Where is JWT validation implemented?" → search: repo code
"Summarize the report" → search: PDFs
```

This improves precision. If you ask a code question, the system does not waste time searching your PDF chunks.

**Step 9b — Hybrid Retrieval**

Two retrievals run in parallel:
- Dense retrieval: BGE embeddings + Qdrant similarity search
- Sparse retrieval: BM25 keyword matching

**Step 9c — Reciprocal Rank Fusion (RRF)**

The two ranked lists are merged using RRF. Each chunk's score is based on its rank in both lists. Chunks that appear high in both lists get the highest combined scores.

**Step 9d — Cross-Encoder Reranking**

The top candidates from RRF are reranked using a cross-encoder model.

Unlike embedding similarity (which compares vectors independently), the cross-encoder reads the query and each chunk *together* and gives a more accurate relevance score.


In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    CONFIG["cross_encoder_model"]
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
def route_query(query: str, chunks: List[Dict]) -> str:

    q = query.lower()

    has_pdf = any(
        c["source"].startswith("pdf:")
        for c in chunks
    )

    has_repo = any(
        c["source"].startswith("repo:")
        for c in chunks
    )

    if any(
        kw in q
        for kw in [
            "code",
            "function",
            "class",
            "method",
            "repository",
            "file",
            "implementation",
        ]
    ):
        return "repo" if has_repo else "all"

    if any(
        kw in q
        for kw in [
            "pdf",
            "document",
            "paper",
            "report",
            "summary",
            "table",
        ]
    ):
        return "pdf" if has_pdf else "all"

    return "all"


def retrieve(
    query: str,
    chunks: List[Dict],
    emb: EmbedEngine,
    qc: QdrantClient,
    bm25: BM25Okapi,
) -> List[Dict]:

    coll = CONFIG["qdrant_collection"]

    K = CONFIG["top_k_retrieve"] * 2

    route = route_query(query, chunks)

    print(
        f"[Router] Route: {route.upper()}"
    )

    # Stage 1A
    q_vec = emb.embed_one(query).tolist()

    dense_hits = qc.query_points(
        collection_name=coll,
        query=q_vec,
        limit=K,
    ).points

    dense_ranks = {
        int(hit.id): rank
        for rank, hit in enumerate(dense_hits)
    }

    # Stage 1B
    bm25_scores = bm25.get_scores(
        query.lower().split()
    )

    top_bm25_idx = np.argsort(
        bm25_scores
    )[::-1][:K]

    bm25_ranks = {
        int(idx): rank
        for rank, idx in enumerate(
            top_bm25_idx
        )
    }

    # Stage 2: RRF
    k_rrf = CONFIG["rrf_k"]

    all_ids = (
        set(dense_ranks)
        | set(bm25_ranks)
    )

    rrf_scores = {

        idx:

        (
            1 / (
                k_rrf +
                dense_ranks.get(idx, K)
            )
        )

        +

        (
            1 / (
                k_rrf +
                bm25_ranks.get(idx, K)
            )
        )

        for idx in all_ids
    }

    sorted_idx = [

        idx

        for idx, _ in sorted(
            rrf_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )
    ]

    # Stage 3: Route Filter

    candidates = []

    for idx in sorted_idx:

        chunk = next(
            (
                c
                for c in chunks
                if c["_idx"] == idx
            ),
            None
        )

        if not chunk:
            continue

        src = chunk["source"]

        if (
            route == "repo"
            and not src.startswith("repo:")
        ):
            continue

        if (
            route == "pdf"
            and not src.startswith("pdf:")
        ):
            continue

        candidates.append(chunk)

        if len(candidates) >= CONFIG["top_k_retrieve"]:
            break

    if not candidates:

        print(
            "[Router] No match. Falling back."
        )

        candidates = [

            c

            for c in chunks

            if c["_idx"]
            in sorted_idx[
                :CONFIG["top_k_retrieve"]
            ]
        ]

    # Stage 4: Cross Encoder Rerank

    pairs = [

        (
            query,
            c["content"]
        )

        for c in candidates

    ]

    scores = reranker.predict(
        pairs
    )

    for c, score in zip(
        candidates,
        scores
    ):
        c["_score"] = float(score)

    candidates.sort(
        key=lambda x: x["_score"],
        reverse=True
    )

    return candidates[
        :CONFIG["top_k_final"]
    ]


print(
    "Knowledge Routing + Hybrid Retrieval + Cross Encoder ready."
)

Knowledge Routing + Hybrid Retrieval + Cross Encoder ready.


## Step 10 — Semantic Cache and Answer Generation

**Semantic Cache**

Before doing any retrieval, the system checks whether a very similar question was already asked.

If a previous answer has similarity above the threshold, it is returned immediately — no embedding, no vector search, no LLM call. This saves time and reduces API costs.

**Dynamic Context Budgeting**

Retrieved chunks are added to the prompt one by one until the token budget is full. This prevents sending too much to the LLM, which would waste tokens and increase cost.

**Answer Generation**

The selected context is sent to Groq (Llama model) with a prompt that says: answer only from the given context, do not make things up, and cite your sources.

**Why I removed CRAG:**
An earlier version had a Corrective RAG step where an LLM graded each retrieved chunk. After adding BGE embeddings, BM25, RRF, and cross-encoder reranking, retrieval quality was already strong enough that the grading step added more cost and latency than value. Better retrieval beats adding more LLM calls.


In [ ]:
import numpy as np
from groq import Groq


def _friendly_source(source: str) -> str:

    if source.startswith("pdf:"):
        return source[4:]

    if source.startswith("repo:"):
        return source[5:]

    if source.startswith("text:"):
        return "Pasted Text"

    return source


def semantic_cache_check(
    query_vec: np.ndarray,
    cache: list,
    threshold: float = 0.95
):

    if not cache:
        return None

    q_norm = query_vec / (
        np.linalg.norm(query_vec) + 1e-9
    )

    best_score = -1.0
    best_answer = None

    for item in cache:

        c_vec = np.array(
            item["query_vec"]
        )

        c_norm = c_vec / (
            np.linalg.norm(c_vec) + 1e-9
        )

        score = float(
            np.dot(q_norm, c_norm)
        )

        if score > best_score:

            best_score = score
            best_answer = item["answer"]

    if best_score >= threshold:

        print(
            f"[CACHE HIT] "
            f"{best_score:.3f}"
        )

        return best_answer

    print(
        f"[CACHE MISS] "
        f"{best_score:.3f}"
    )

    return None


def generate_answer(
    query: str,
    chunks: list,
    api_key: str
):

    client = Groq(
        api_key=api_key
    )

    context_budget = CONFIG[
        "context_budget"
    ]

    used_tokens = 0

    context_parts = []

    sources = []

    for c in chunks:

        chunk_tokens = c.get(
            "tokens",
            0
        )

        if (
            used_tokens +
            chunk_tokens
            >
            context_budget
        ):
            break

        used_tokens += chunk_tokens

        src = _friendly_source(
            c.get(
                "source",
                "unknown"
            )
        )

        lines = c.get(
            "lines",
            ""
        )

        location = (
            f"{src}:{lines}"
            if lines
            else src
        )

        context_parts.append(

            f"[SOURCE: {location}]\n"
            f"{c['content']}"

        )

        sources.append({

            "source": src,

            "location": lines,

            "type": c.get(
                "type",
                "unknown"
            )

        })

    context_text = "\n\n".join(
        context_parts
    )

    prompt = f"""
You are an expert technical assistant.

Use ONLY the provided context.

If the answer cannot be found in the context,
say:

"The context does not provide an answer."

Always mention supporting sources.

CONTEXT:

{context_text}

QUESTION:

{query}
"""

    completion = (
        client.chat.completions.create(

            model=CONFIG[
                "groq_model"
            ],

            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],

            temperature=0.2,

            max_tokens=1024,
        )
    )

    answer = (
        completion
        .choices[0]
        .message.content
        .strip()
    )

    return {

        "answer": answer,

        "sources": sources,

        "context_tokens":
            used_tokens
    }


print(
    "Answer Generator + Semantic Cache ready."
)

Answer Generator + Semantic Cache ready.


## Step 11 — Gradio Interface

This cell launches the interactive web interface.

The interface has three input sections:
- **GitHub URL** — paste a public repo URL and click Add Repo
- **PDF upload** — upload one or more PDF files
- **Text paste** — paste any notes or documentation

After adding sources, click **Build Index**. Then type a question and click Ask.

The answer shows which source it came from and which route the query took.


In [ ]:
# Runtime state used during indexing and retrieval
STATE: Dict[str, Any] = {
    "chunks" : [],
    "emb"    : None,
    "qc"     : None,
    "bm25"   : None,
    "ready"  : False,
    "api_key": "",
}

def ui_ingest(api_key_in, repo_url, pdf_file, md_text, progress=gr.Progress()):
    global STATE
    STATE["ready"] = False

    api_key = (api_key_in or "").strip()
    if not api_key:
        return "API key required.", gr.update(interactive=False)
    STATE["api_key"] = api_key

    has_repo = bool(repo_url and repo_url.strip())
    has_pdf  = pdf_file is not None
    has_text = bool(md_text and md_text.strip())

    if not (has_repo or has_pdf or has_text):
        return "Provide at least one source (GitHub URL, PDF, or text).", gr.update(interactive=False)

    all_chunks = []
    try:
        if has_repo:
            progress(0.10, desc="Cloning repository")
            all_chunks += ingest_github_repo(repo_url.strip())

        if has_pdf:
            progress(0.40, desc="Parsing PDF")
            all_chunks += ingest_pdf(pdf_file.name)

        if has_text:
            progress(0.60, desc="Chunking text")
            all_chunks += ingest_text(md_text, label="markdown")

        if not all_chunks:
            return "No content extracted.", gr.update(interactive=False)

        progress(0.75, desc="Building indexes")
        emb, qc, bm25 = build_index(all_chunks)
        STATE.update({"chunks": all_chunks, "emb": emb, "qc": qc, "bm25": bm25, "ready": True})

        parts = []
        code_n = sum(1 for c in all_chunks if c["type"] == "code")
        pdf_n  = sum(1 for c in all_chunks if c["type"] == "pdf")
        txt_n  = sum(1 for c in all_chunks if c["type"] == "text")
        if code_n: parts.append(f"Code: {code_n}")
        if pdf_n : parts.append(f"PDF: {pdf_n}")
        if txt_n : parts.append(f"Text: {txt_n}")

        progress(1.0, desc="Complete")
        return f"Ready. Indexed {len(all_chunks)} chunks. (" + ", ".join(parts) + ")", gr.update(interactive=True)
    except Exception:
        import traceback
        return f"Error during ingestion:\n{traceback.format_exc()}", gr.update(interactive=False)

def ui_query(question, show_trace):
    if not STATE["ready"]:
        return "Index not built.", ""
    if not question.strip():
        return "Query cannot be empty.", ""
    try:
        t0 = time.time()
        top_chunks = retrieve(question, STATE["chunks"], STATE["emb"], STATE["qc"], STATE["bm25"])
        if not top_chunks:
            return "No matching context found.", ""

        result  = generate_answer(question, top_chunks, STATE["api_key"])
        elapsed = time.time() - t0

        answer_md = (
            f"### Response\n\n{result['answer']}\n\n"
            f"---\n"
            f"**Metrics**\n"
            f"- Context reduction: {result['saved_pct']:.1f}% saved\n"
            f"- Token usage: {result['total_tokens']:,} tokens (vs naive {result['total_tokens'] + result['saved_tokens']:,})\n"
            f"- Retrieval latency: {elapsed:.2f}s"
        )

        trace_md = format_trace(top_chunks) if show_trace else ""
        return answer_md, trace_md

    except Exception:
        import traceback
        return f"Error executing query:\n```\n{traceback.format_exc()}\n```", ""

with gr.Blocks(theme=gr.themes.Default()) as demo:
    gr.Markdown("# TraceIQ - Code Search")
    gr.Markdown("Multi-source retrieval over code, PDFs, and text.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 1. Ingestion")
            api_key_box = gr.Textbox(
                label="Groq API Key",
                type="password"
            )
            repo_box = gr.Textbox(
                label="GitHub Repository URL",
                placeholder="https://github.com/tiangolo/fastapi"
            )
            pdf_box = gr.File(label="Upload PDF")
            md_box  = gr.Textbox(
                label="Paste Text",
                lines=5
            )
            ingest_btn    = gr.Button("Build Index", variant="primary")
            ingest_status = gr.Textbox(label="Status", interactive=False, lines=3)

        with gr.Column(scale=2):
            gr.Markdown("### 2. Search")
            with gr.Row():
                question_box = gr.Textbox(
                    label="Query",
                    lines=2,
                    scale=5
                )
                trace_toggle = gr.Checkbox(label="Show Context", value=True, scale=1)
            ask_btn = gr.Button("Execute Query", variant="secondary", interactive=False)
            gr.Markdown("---")
            answer_out = gr.Markdown()
            trace_out  = gr.Markdown()

    ingest_btn.click(
        fn=ui_ingest,
        inputs=[api_key_box, repo_box, pdf_box, md_box],
        outputs=[ingest_status, ask_btn],
    )
    ask_btn.click(
        fn=ui_query,
        inputs=[question_box, trace_toggle],
        outputs=[answer_out, trace_out],
    )
    question_box.submit(
        fn=ui_query,
        inputs=[question_box, trace_toggle],
        outputs=[answer_out, trace_out],
    )

demo.launch(share=True)



/tmp/ipykernel_491/1932746394.py:139: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(


✅ Gradio UI built. Run the next cell to launch.


## Step 12 — Launch

Run this cell to start the Gradio interface.

A public URL will appear that you can share with others.


In [ ]:
# Runtime state

STATE = {
    "chunks": [],
    "emb": None,
    "qc": None,
    "bm25": None,
    "ready": False,
    "api_key": "",
}


def ui_ingest(
    api_key_in,
    repo_url,
    pdf_files,
    md_text,
    progress=gr.Progress()
):

    global STATE

    STATE["ready"] = False

    api_key = (api_key_in or "").strip()

    if not api_key:
        return (
            "Groq API key required.",
            gr.update(interactive=False)
        )

    STATE["api_key"] = api_key

    has_repo = bool(
        repo_url and repo_url.strip()
    )

    has_pdf = bool(pdf_files)

    has_text = bool(
        md_text and md_text.strip()
    )

    if not (
        has_repo
        or has_pdf
        or has_text
    ):
        return (
            "Provide at least one source.",
            gr.update(interactive=False)
        )

    all_chunks = []

    try:

        if has_repo:

            progress(
                0.15,
                desc="Cloning repository"
            )

            all_chunks.extend(
                ingest_github_repo(
                    repo_url.strip()
                )
            )

        if has_pdf:

            progress(
                0.45,
                desc="Processing PDFs"
            )

            pdf_paths = [
                f.name
                for f in pdf_files
            ]

            all_chunks.extend(
                ingest_pdfs(
                    pdf_paths
                )
            )

        if has_text:

            progress(
                0.65,
                desc="Processing text"
            )

            all_chunks.extend(
                ingest_text(
                    md_text,
                    label="markdown"
                )
            )

        if not all_chunks:

            return (
                "No content extracted.",
                gr.update(interactive=False)
            )

        progress(
            0.80,
            desc="Building indexes"
        )

        emb, qc, bm25 = build_index(
            all_chunks
        )

        STATE.update({

            "chunks": all_chunks,

            "emb": emb,

            "qc": qc,

            "bm25": bm25,

            "ready": True

        })

        code_chunks = sum(
            1
            for c in all_chunks
            if c["type"] == "code"
        )

        pdf_chunks = sum(
            1
            for c in all_chunks
            if c["type"] == "pdf"
        )

        text_chunks = sum(
            1
            for c in all_chunks
            if c["type"] == "text"
        )

        total_tokens = sum(
            c["tokens"]
            for c in all_chunks
        )

        progress(
            1.0,
            desc="Complete"
        )

        status = (
            f"Index Ready\n\n"
            f"Total Chunks: {len(all_chunks)}\n"
            f"Code Chunks: {code_chunks}\n"
            f"PDF Chunks: {pdf_chunks}\n"
            f"Text Chunks: {text_chunks}\n"
            f"Total Tokens: {total_tokens:,}"
        )

        return (
            status,
            gr.update(interactive=True)
        )

    except Exception:

        import traceback

        return (
            traceback.format_exc(),
            gr.update(interactive=False)
        )


def ui_query(
    question,
    show_trace
):

    if not STATE["ready"]:

        return (
            "Build the index first.",
            ""
        )

    if not question.strip():

        return (
            "Query cannot be empty.",
            ""
        )

    try:

        t0 = time.time()

        top_chunks = retrieve(
            question,
            STATE["chunks"],
            STATE["emb"],
            STATE["qc"],
            STATE["bm25"]
        )

        if not top_chunks:

            return (
                "No relevant context found.",
                ""
            )

        result = generate_answer(
            question,
            top_chunks,
            STATE["api_key"]
        )

        elapsed = (
            time.time() - t0
        )

        answer_md = (
            f"## Answer\n\n"
            f"{result['answer']}\n\n"
            f"---\n"
            f"### Metrics\n"
            f"- Context Sent: {result['context_tokens']:,} tokens\n"
            f"- Retrieval Latency: {elapsed:.2f}s\n"
            f"- Retrieved Chunks: {len(top_chunks)}"
        )

        trace_md = ""

        if show_trace:

            rows = []

            for i, c in enumerate(
                top_chunks,
                start=1
            ):

                rows.append(

                    f"### {i}. {c['source']}\n"
                    f"Type: {c['type']}\n"
                    f"Name: {c['name']}\n"
                    f"Score: {c.get('_score',0):.4f}\n\n"
                    f"```text\n"
                    f"{c['content'][:500]}"
                    f"\n```"

                )

            trace_md = "\n\n".join(
                rows
            )

        return (
            answer_md,
            trace_md
        )

    except Exception:

        import traceback

        return (
            f"Error:\n```text\n{traceback.format_exc()}\n```",
            ""
        )


with gr.Blocks(
    title="TraceIQ Advanced RAG"
) as demo:

    gr.Markdown(
        "# TraceIQ Advanced RAG"
    )

    gr.Markdown(
        """
Hybrid Retrieval using:

- BGE Embeddings
- Qdrant Vector Search
- BM25 Keyword Search
- RRF Fusion
- Cross Encoder Reranking
- Dynamic Context Budgeting

Supports:

- GitHub Repositories
- Multiple PDFs
- Pasted Text
"""
    )

    with gr.Row():

        with gr.Column(scale=1):

            gr.Markdown(
                "## Ingestion"
            )

            api_key_box = gr.Textbox(
                label="Groq API Key",
                type="password"
            )

            repo_box = gr.Textbox(
                label="GitHub Repository URL",
                placeholder="https://github.com/..."
            )

            pdf_box = gr.File(
                label="Upload PDFs",
                file_count="multiple"
            )

            md_box = gr.Textbox(
                label="Paste Text",
                lines=6
            )

            ingest_btn = gr.Button(
                "Build Index",
                variant="primary"
            )

            ingest_status = gr.Textbox(
                label="Status",
                lines=8,
                interactive=False
            )

        with gr.Column(scale=2):

            gr.Markdown(
                "## Search"
            )

            question_box = gr.Textbox(
                label="Question",
                lines=2
            )

            trace_toggle = gr.Checkbox(
                label="Show Retrieved Context",
                value=True
            )

            ask_btn = gr.Button(
                "Ask",
                interactive=False
            )

            answer_out = gr.Markdown()

            with gr.Accordion(
                "Retrieved Chunks",
                open=False
            ):
                trace_out = gr.Markdown()

    ingest_btn.click(
        fn=ui_ingest,
        inputs=[
            api_key_box,
            repo_box,
            pdf_box,
            md_box
        ],
        outputs=[
            ingest_status,
            ask_btn
        ]
    )

    ask_btn.click(
        fn=ui_query,
        inputs=[
            question_box,
            trace_toggle
        ],
        outputs=[
            answer_out,
            trace_out
        ]
    )

    question_box.submit(
        fn=ui_query,
        inputs=[
            question_box,
            trace_toggle
        ],
        outputs=[
            answer_out,
            trace_out
        ]
    )

demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d3a3010b626be7b4bb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
